執行sql/01_data_cleaning.sql前需先在 BigQuery 建立目標 dataset：CREATE SCHEMA IF NOT EXISTS \traffic_ad_roi_clean`

In [ ]:
CREATE SCHEMA IF NOT EXISTS traffic_ad_roi_clean;

| 步驟                           | 目的                           |
| ---------------------------- | ---------------------------- |
| Step 1 — Validation Checks   | 執行前先檢查，輸出 QA 報表              |
| Step 2 — Clean & Standardise | 清洗後寫入 traffic_ad_roi_clean.* |
| Step 3 — Summary Report      | 對比 raw vs clean 行數，確認結果      |

清洗邏輯說明
每張表均涵蓋以下清洗處理：

去重：ROW_NUMBER() OVER (PARTITION BY <pk>) 保留最新一筆，處理重複 primary key

NULL / 空值處理：COALESCE + NULLIF(TRIM(...), '') 統一填補預設值

Channel 標準化：CASE UPPER(TRIM(channel)) 將 GOOGLE、GOOGLE ADS 等變體統一為受控詞彙，對應你的 campaigns 表原有值

負數數值修正：GREATEST(..., 0) 防止 impressions、clicks、spend_usd 出現負數

CTR 重算：從原始 clicks/impressions 重新計算，比直接信任儲存值更可靠

衍生欄位：新增 is_active（campaigns）、cost_per_click_usd（ad_impressions）、engagement_tier（sessions）、order_value_tier（conversions）方便下游分析